In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_124.jpg
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_949.jpg
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_786.jpg
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_371.jpg
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_599.jpg
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_802.jpg
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_1323.jpg
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_1347.jpg
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_955.jpg
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training/pituitary/Tr-pi_778.jpg
/kaggle/input/datasets/masou

In [1]:
pip install torch torchvision timm

Note: you may need to restart the kernel to use updated packages.


In [21]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(
    "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing",
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16
)

In [26]:
import torch
import torch.nn as nn

In [27]:
import timm

model = timm.create_model(
    "swin_base_patch4_window7_224",
    pretrained=True,
    num_classes=4
)

In [28]:
for param in model.parameters():
    param.requires_grad = False

for param in model.head.parameters():
    param.requires_grad = True

In [7]:
criterion = torch.nn.CrossEntropyLoss()

In [8]:
optimizer = torch.optim.AdamW(
    model.head.parameters(),
    lr=1e-3
)

In [20]:
sum([p.numel() for p in model.parameters()  ])

86747324

In [29]:
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

Using 2 GPUs


In [13]:
from tqdm.auto import tqdm

def train_one_epoch(model, loader):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc="Training", leave=False)

    for images, labels in pbar:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Live update
        pbar.set_postfix(
            loss=f"{total_loss/(pbar.n+1):.4f}",
            acc=f"{100*correct/total:.2f}%"
        )

    loss = total_loss / len(loader)
    acc = correct / total

    return loss, acc

In [14]:
def evaluate(model, loader):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            _, predicted = torch.max(outputs,1)

            correct += (predicted==labels).sum().item()

            total += labels.size(0)

    loss = total_loss / len(loader)

    acc = correct / total

    return loss, acc

In [16]:
epochs = 30
best_model = float("inf")

for epoch in range(epochs):

    print(f"\nEpoch {epoch+1}/{epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader)

    val_loss, val_acc = evaluate(model, val_loader)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.4f}")
    print("-"*40)

    if val_loss < best_model:
        best_model = val_loss
        torch.save(model.state_dict(), "vit_model.pth")
        print("✅ Best model saved!")


Epoch 1/5


Training:   0%|          | 0/350 [00:00<?, ?it/s]

Train Loss : 0.4515
Train Acc  : 0.8512
Val Loss   : 0.6019
Val Acc    : 0.8050
----------------------------------------
✅ Best model saved!

Epoch 2/5


Training:   0%|          | 0/350 [00:00<?, ?it/s]

Train Loss : 0.3868
Train Acc  : 0.8738
Val Loss   : 0.5699
Val Acc    : 0.8225
----------------------------------------
✅ Best model saved!

Epoch 3/5


...

Train Loss : 0.3602
Train Acc  : 0.8759
Val Loss   : 0.5642
Val Acc    : 0.8294
----------------------------------------
✅ Best model saved!

Epoch 4/5


Training:   0%|          | 0/350 [00:00<?, ?it/s]

Train Loss : 0.3353
Train Acc  : 0.8823
Val Loss   : 0.5463
Val Acc    : 0.8419
----------------------------------------
✅ Best model saved!

Epoch 5/5


Training:   0%|          | 0/350 [00:00<?, ?it/s]

Train Loss : 0.3103
Train Acc  : 0.8996
Val Loss   : 0.5315
Val Acc    : 0.8438
----------------------------------------
✅ Best model saved!


In [17]:
evaluate(model,val_loader)

(0.5314731343649328, 0.84375)

In [18]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        pred = outputs.argmax(1)

        correct += (pred == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

In [19]:
accuracy

0.84375

In [ ]:
h